# Extension 4 — Conclusions and Project Guide

## 1. How to read this project

| # | Story step | Notebook(s) |
|---|---|---|
| 1 | Problem, objectives, decision-support motivation | [Demonstration](Overview_00_Project_Demonstration.ipynb), [Phase 1](Phase_01_Project_and_Data_Understanding.ipynb) |
| 2 | Data sources, collection, cleaning, quality, limitations | [Phase 2](Phase_02_Crop_Feasibility_Audit_and_Selection.ipynb), [Phase 3](Phase_03_Weather_Data_Collection.ipynb), [Phase 4](Phase_04_Data_Integration.ipynb), [Phase 5](Phase_05_Data_Cleaning.ipynb), [Phase 6](Phase_06_Feature_Engineering.ipynb), [Extension 1](Research_Extension_01_Candidate_Price_Data.ipynb) |
| 3 | Exploratory analysis | [Phase 7](Phase_07_EDA.ipynb) |
| 4 | Targets, evaluation method, persistence baselines | [Phase 8](Phase_08_Price_Forecasting.ipynb), [Extension 2 §1–2](Research_Extension_02_Candidate_Forecast_Experiments.ipynb) |
| 5 | Learned models and why they did not beat persistence | [Phase 8](Phase_08_Price_Forecasting.ipynb), [Extension 2 §3](Research_Extension_02_Candidate_Forecast_Experiments.ipynb); arrivals: [Phase 9](Phase_09_Arrivals_Model_Experiments.ipynb) |
| 6 | Weather experiment and why weather was not adopted | [Extension 2 §4](Research_Extension_02_Candidate_Forecast_Experiments.ipynb) |
| 7 | Final persistence test (D-52) and its uncertainty | [Extension 3](Research_Extension_03_Final_Persistence_Test.ipynb) |
| 8 | Conclusions, usefulness, limitations, future work | this notebook; production context: [Phase 10](Phase_10_Production_Context.ipynb) |

**Production vs candidate.** Phases 1–10 are the production pipeline behind the application. Extensions 1–3 are a separately versioned **candidate** experiment on 2024–2026 data; none of it is a production model.

In [1]:
# Setup: locate the project root; read saved outputs only (no parquet, no network, no writes)
import hashlib, json, os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 140)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
C = Path("Processed_Data/candidates/datagov_prices")      # candidate data and experiments (not production)
EXP = C / "experiments"
CROPS = ["Onion", "Potato", "Wheat"]
COLORS = {"Onion": "#7b3294", "Potato": "#c2a5cf", "Wheat": "#e66101"}
print("Python", sys.version.split()[0])

Python 3.13.7


## 2. The evidence in one table

Price forecasting, one week ahead, MAE relative to persistence (negative skill = worse than persistence):

In [2]:
learned = pd.read_csv(EXP / "exp_candidate_v1/learned_hgb_conservative_faq_only_h1/run/by_crop.csv")
weather = pd.read_csv(EXP / "exp_candidate_weather_v1/run/by_crop.csv")
test = pd.read_csv(EXP / "exp_candidate_v1/persistence_test_d52/run/by_crop.csv")
summary = pd.DataFrame({
    "learned model skill (validation)": learned[learned.model == "learned"].set_index("Commodity").skill_vs_persistence,
    "price+weather skill (validation, weather subset)": weather[weather.model == "price_weather"].set_index("Commodity").skill_vs_persistence,
    "persistence test MAE (Rs/q)": test[(test.design == "faq_only") & (test.horizon == 1)].set_index("Commodity").mae,
    "persistence test MAPE %": test[(test.design == "faq_only") & (test.horizon == 1)].set_index("Commodity").mape_pct})
summary.round({"learned model skill (validation)": 3, "price+weather skill (validation, weather subset)": 3,
               "persistence test MAE (Rs/q)": 1, "persistence test MAPE %": 1})

,learned model skill (validation),"price+weather skill (validation, weather subset)",persistence test MAE (Rs/q),persistence test MAPE %
Commodity,,,,
Onion,-0.364,-0.340,231.3,7.6
Potato,-0.154,-0.242,32.0,3.5
Wheat,-0.096,0.007,40.7,1.5


## 3. Conclusions

1. **Prices are highly persistent.** On both the original data (Phase 8, 2021–2022 test) and the 2024–2026 candidate data, no learned model beat "next week = this week" in a pre-registered, one-time test.
2. **Arrivals are different.** The Phase 9 learned arrivals model beat persistence in all 18 test cell-years (by 2.3–13.5% lower log MAE). Arrivals measure supply activity, not demand.
3. **Weather did not help** price forecasting on the subset where it exists, and it is not used.
4. **Honest uncertainty matters.** The persistence test error is small for Wheat and Potato (1.5–3.5% MAPE one week ahead) and larger for Onion (7.6%, rising to 21% at four weeks), with wider uncertainty across states than across markets.

## 4. Practical usefulness

- A trader can treat this week's mandi price as next week's expectation, and use the Extension 3 error bands (by crop, horizon and month) to size the risk, especially for Onion in late summer.
- The application shows prices, arrivals, production context, road routes and mapped warehouses side by side, with every data boundary enforced.
- The pipeline shows how to evaluate forecasts without fooling oneself: frozen protocols, run-once tests, cluster-robust intervals and recorded negative results.

## 5. Limitations

- Candidate data covers only ~2.7 years, price only, concentrated in a few states (Wheat: 5).
- The persistence test covers 16 weeks of one season; weeks without observed target prices are absent.
- The 2023 holdout is still unscored. Routing has no transport cost, and warehouses have no verified capacity or tariffs.
- Spark runs on one machine (local mode), not a cluster.

## 6. Future work

1. **Lawful market-level arrivals** for the candidate period: the DMI request is drafted but not sent; CEDA terms are only partly verified.
2. **A fresh evaluation period** (data after 2026-10-20) for any new price or arrivals model, with a new pre-registration.
3. Score the reserved 2023 holdout once, under its existing protocol, if approved.
4. Transport costs and verified warehouse data, then the planned LP/MILP decision model.